# Yambda → inter.json + content_embeddings.pkl

Аналог `notebooks/VkDatasetProcessing.ipynb` для датасета [yandex/yambda](https://huggingface.co/datasets/yandex/yambda). На выходе:
* `../data/yambda/inter.json` — `{str(user_id): [item_id, ...]}`, dense 0-index, сорт по времени
* `../data/yambda/content_embeddings.pkl` — `{'item_id': [...], 'embedding': [...]}` (D=64, normalized_embed)

Исходный объём flat/50m/likes слишком большой для TIGER, поэтому сразу в том же ноутбуке жмём по VK_small-схеме: random-сабсэмпл юзеров + truncation до последних K интеракций.

Подробное обоснование цифр — в [ai/vk_exps/vk_plan4_smaller_dataset.md](../ai/vk_exps/vk_plan4_smaller_dataset.md) и [ai/vk_exps/task5.txt](../ai/vk_exps/task5.txt).

In [ ]:
%pip install polars==1.36.1 numpy pyarrow huggingface_hub datasets

In [ ]:
import json
import os
import pickle
import random

import numpy as np
import polars as pl

## Конфигурация

* `DATASET_PATH` — куда скачивать HF-артефакты. На A100-сервере под jovyan, на других машинах — переопределить.
* `USER_SUBSAMPLE_RATIO` и `MAX_HISTORY_PER_USER` — рычаги сжатия. Дефолт подобран так, чтобы получить ~50k юзеров / ~500k интеракций — как `data/VK_small/`.

In [ ]:
DATASET_TYPE = 'flat'
DATASET_SIZE = '50m'
EVENT = 'likes'

DATASET_PATH = '/home/jupyter/project/tiger-cf/yambda'
OUTPUT_DIR = '../data/yambda'

CORE_K = 5
USER_SUBSAMPLE_RATIO = 0.05
MAX_HISTORY_PER_USER = 15
RANDOM_SEED = 42

INTERACTIONS_OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'inter.json')
EMBEDDINGS_OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'content_embeddings.pkl')

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Скачивание эмбедов и интеракций

Скачиваем `embeddings.parquet` и `flat/50m/likes.parquet` через `hf_hub_download` (как в [ai/vk_exps/YambdaDownload.ipynb](../ai/vk_exps/YambdaDownload.ipynb)). Если файлы уже в `DATASET_PATH` — `hf_hub_download` просто вернёт путь из кэша.

In [ ]:
from huggingface_hub import hf_hub_download

os.environ['HF_ENDPOINT'] = os.environ.get('HF_ENDPOINT', 'https://huggingface.co')

embeddings_path = hf_hub_download(
    repo_id='yandex/yambda',
    repo_type='dataset',
    filename='embeddings.parquet',
    local_dir=DATASET_PATH,
)
print(f'embeddings: {embeddings_path}')

interactions_filename = f'{DATASET_TYPE}/{DATASET_SIZE}/{EVENT}.parquet'
interactions_path = hf_hub_download(
    repo_id='yandex/yambda',
    repo_type='dataset',
    filename=interactions_filename,
    local_dir=DATASET_PATH,
)
print(f'interactions: {interactions_path}')

## Чтение интеракций

Yambda flat/likes уже отфильтрован по типу события, так что timespent-фильтра (как в VK) не нужно. Переименовываем `uid → user_id`, выбираем только нужные колонки.

In [ ]:
all_inter = (
    pl.read_parquet(interactions_path)
    .rename({'uid': 'user_id'})
    .select(['user_id', 'item_id', 'timestamp'])
)
print('всего интеракций:', all_inter.shape)

## Пересечение с эмбедами

Оставляем только интеракции с айтемами, у которых есть вектор в `embeddings.parquet`.

In [ ]:
emb_table = pl.read_parquet(embeddings_path)
print('embeddings columns:', emb_table.columns)
emb_item_ids = emb_table['item_id'].to_numpy()
emb_vectors_raw = emb_table['normalized_embed'].to_numpy()
print('эмбедов всего:', emb_item_ids.shape)

In [ ]:
items_with_emb = pl.DataFrame({'item_id': emb_item_ids})
filtered_df = all_inter.join(items_with_emb, on='item_id', how='inner')
print('после пересечения с эмбедами:', filtered_df.shape)

## Random-сабсэмпл юзеров

Yambda 50m слишком большой для TIGER без сжатия. Берём фиксированную долю юзеров (дефолт 5%) — это даёт сравнимый с VK_small объём.

In [ ]:
if USER_SUBSAMPLE_RATIO < 1.0:
    unique_users_full = filtered_df['user_id'].unique().to_numpy()
    rng = np.random.default_rng(RANDOM_SEED)
    keep_n = int(len(unique_users_full) * USER_SUBSAMPLE_RATIO)
    keep_users = rng.choice(unique_users_full, size=keep_n, replace=False)
    keep_users_df = pl.DataFrame({'user_id': keep_users})
    filtered_df = filtered_df.join(keep_users_df, on='user_id', how='inner')
    print(f'после сабсэмпла юзеров ({USER_SUBSAMPLE_RATIO}): {filtered_df.shape}')
else:
    print('USER_SUBSAMPLE_RATIO=1.0 — берём всех юзеров')
    print('всего:', filtered_df.shape)

## Truncation: последние K интеракций на юзера

Оставляем последние `MAX_HISTORY_PER_USER` интеракций по таймстампу. `MAX_SEQ_LEN=20` в TIGER и так обрезает до 20 элементов на forward, поэтому на качество это почти не влияет, а число train-сэмплов (при `is_extended=True`) падает линейно.

In [ ]:
if MAX_HISTORY_PER_USER is not None:
    filtered_df = (
        filtered_df
        .sort(['user_id', 'timestamp'])
        .with_columns(
            pl.col('timestamp').rank('ordinal', descending=True).over('user_id').alias('rank_desc')
        )
        .filter(pl.col('rank_desc') <= MAX_HISTORY_PER_USER)
        .drop('rank_desc')
    )
    print(f'после truncation (K={MAX_HISTORY_PER_USER}): {filtered_df.shape}')

## Core-5 фильтрация

Итеративно выкидываем юзеров и айтемы с <5 интеракций до сходимости.

In [ ]:
is_changed = True
iteration = 0
while is_changed:
    iteration += 1
    user_counts = filtered_df.group_by('user_id').agg(pl.len().alias('user_count'))
    item_counts = filtered_df.group_by('item_id').agg(pl.len().alias('item_count'))

    good_users = user_counts.filter(pl.col('user_count') >= CORE_K).select('user_id')
    good_items = item_counts.filter(pl.col('item_count') >= CORE_K).select('item_id')

    old_size = len(filtered_df)
    new_df = filtered_df.join(good_users, on='user_id', how='inner')
    new_df = new_df.join(good_items, on='item_id', how='inner')
    new_size = len(new_df)

    filtered_df = new_df
    is_changed = old_size != new_size
    print(f'iter {iteration}: {old_size} -> {new_size}')

print('финал после Core-5:', filtered_df.shape)

## Ремап user_id и item_id в 0-indexed dense

In [ ]:
unique_users = filtered_df['user_id'].unique(maintain_order=True).to_list()
user_ids_mapping = {value: i for i, value in enumerate(unique_users)}

unique_items = filtered_df['item_id'].unique(maintain_order=True).to_list()
item_ids_mapping = {value: i for i, value in enumerate(unique_items)}

num_users = len(user_ids_mapping)
num_items = len(item_ids_mapping)
print('num_users:', num_users, 'num_items:', num_items)

In [ ]:
filtered_df = filtered_df.with_columns([
    pl.col('user_id').replace_strict(user_ids_mapping).alias('user_id'),
    pl.col('item_id').replace_strict(item_ids_mapping).alias('item_id'),
])
filtered_df.head()

## Группировка по юзеру и сериализация inter.json

In [ ]:
filtered_df = filtered_df.sort(['user_id', 'timestamp'])
grouped = (
    filtered_df
    .group_by('user_id', maintain_order=True)
    .agg(pl.col('item_id'))
)
grouped.head()

In [ ]:
json_data = {}
for user_id, item_list in grouped.iter_rows():
    json_data[int(user_id)] = list(map(int, item_list))

assert all(len(v) >= CORE_K for v in json_data.values()), 'Core-5 broken после ремапа'
assert max(max(v) for v in json_data.values()) == num_items - 1
assert min(min(v) for v in json_data.values()) == 0

with open(INTERACTIONS_OUTPUT_PATH, 'w') as f:
    json.dump(json_data, f)

print(f'inter.json: {len(json_data)} юзеров, диапазон item_id [0, {num_items - 1}]')

## Сохранение content_embeddings.pkl

Переупорядочиваем `embeddings.parquet[normalized_embed]` под новый item_id mapping. Формат — как в Amazon/VK: `dict('item_id' -> list[int], 'embedding' -> list[np.ndarray(D,) float32])`.

In [ ]:
old_to_new = item_ids_mapping  # {old_item_id: new_item_id}

old_id_to_pos = {int(old): i for i, old in enumerate(emb_item_ids)}

first_vec = np.asarray(emb_vectors_raw[0], dtype=np.float32)
D = first_vec.shape[0]

new_item_ids = list(range(num_items))
new_embeddings = np.zeros((num_items, D), dtype=np.float32)
for old_id, new_id in old_to_new.items():
    new_embeddings[new_id] = np.asarray(emb_vectors_raw[old_id_to_pos[int(old_id)]], dtype=np.float32)

print('new_embeddings.shape:', new_embeddings.shape)
print('non-zero rows:', int(np.any(new_embeddings != 0, axis=1).sum()))

In [ ]:
out = {
    'item_id': new_item_ids,
    'embedding': [new_embeddings[i] for i in range(num_items)],
}
with open(EMBEDDINGS_OUTPUT_PATH, 'wb') as f:
    pickle.dump(out, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'content_embeddings.pkl сохранён: {EMBEDDINGS_OUTPUT_PATH}')
print(f'  num_items: {len(out["item_id"])}, D: {out["embedding"][0].shape[0]}')

## Финальные sanity-чеки

In [ ]:
with open(INTERACTIONS_OUTPUT_PATH) as f:
    inter = json.load(f)
with open(EMBEDDINGS_OUTPUT_PATH, 'rb') as f:
    emb = pickle.load(f)

assert set(map(int, inter.keys())) == set(range(len(inter)))
assert emb['item_id'] == list(range(len(emb['item_id'])))
assert max(max(v) for v in inter.values()) == len(emb['item_id']) - 1

extended_samples = sum(max(0, len(v) - 3) for v in inter.values())
print('OK:', len(inter), 'юзеров,', len(emb['item_id']), 'айтемов,', emb['embedding'][0].shape, 'эмбед')
print(f'TIGER train samples (is_extended=True): {extended_samples:,}')